# 第 12 章　面向对象编程（下）

> **本章定位**：用**继承**表达"共同点 + 差异点"，用**多态**让调用方不必关心具体类型。这是交通仿真、收费系统、设备管理的基础设计。
>
> **知识点范围**：封装（私有属性 + 公有方法）、继承（单继承、`super()`）、方法重写、多态（同一接口不同实现）、`isinstance()` 类型判断、`NotImplementedError` 约定接口。
>
> **本章约定**：可用前 11 章全部内容；**不引入**抽象基类 `abc`、`@property` 等进阶语法。

---

## 本章学习路径

| 阶段 | 内容 | 编号 | 作用 |
|---|---|---|---|
| 第一部分 | 知识讲解 | 知识点 12.1 — 12.5 | 封装、继承、重写、多态 + 随堂示例 |
| 第二部分 | 代码示范 | 案例 12-1、案例 12-2 | 车型继承 / 设备多态 |
| 第三部分 | 布置练习 | 练习 12-1、练习 12-2 | 车道继承 / 充电桩多态 |
| 第四部分 | 习题答案 | 答案 12-1、答案 12-2 | 对照解析与评分要点 |

**知识点与代码对照表**：

| 知识点 | 名称 | 讲解位置 | 代码示范 | 对应练习 |
|---|---|---|---|---|
| 12.1 | 封装回顾：私有属性 + 公有方法 | 12.2 | 案例 12-1、案例 12-2 | 练习 12-1 |
| 12.2 | 继承与 `super()` | 12.3 | 案例 12-1、案例 12-2 | 练习 12-1 |
| 12.3 | 方法重写 | 12.4 | 案例 12-1 | 练习 12-1、练习 12-2 |
| 12.4 | 多态与 `NotImplementedError` 约定 | 12.5 | 案例 12-2 | 练习 12-2 |
| 12.5 | `isinstance()` 与 `type()` | 12.6 | 案例 12-1 | —— |

## 教学提示（教师用）

- **建议课时**：4 学时（2 次课），对应课程标准模块 12。
- **讲解主线**：继承解决"重复代码"，多态解决"分支爆炸"。讲清 `super()` 是"复用父类实现"而非"替代父类"；多态的关键是"调用同一接口，各自执行不同实现"。
- **学生常见卡点**：子类 `__init__` 忘记调 `super().__init__()`，父类属性缺失；重写时方法签名与父类不一致；`isinstance` 与 `type` 的区别（父类引用可指向子类对象）；基类方法未实现就被调用。
- **课堂活动建议**：给 12-2 增加一种新设备（如可变限速标志），验证主调度循环**一行不改**即可支持，用事实说明多态的价值。

---

# 第一部分　知识讲解

## 12.1 场景导入：车型不同，能不能不写四遍 if

车型不同，限速不同，费率也不同：小客车限速 120、货车限速 80 且费率上浮、客车限速 100。若用 `if vtype == "货车"` 逐个判断，每增加一种车型就要改一遍所有分支。

继承让"差异"只写在子类里，多态让调用方**不必关心具体车型**——同一句 `v.fee(120)`，货车自动算 1.8 倍、客车自动算 1.2 倍，主程序一个字都不用改。这正是交通仿真、收费系统、设备管理的基础设计。

## 12.2 知识点 12.1　封装回顾

### 12.2.1 核心概念

第 11 章已经建立封装的基本手法：

| 手法 | 作用 |
|---|---|
| `self.__x` 私有属性 | 限制外部直接改值 |
| `get_x()` / `set_x()` 公有方法 | 提供受控读写入口 |
| 校验写在 `set_x()` 内 | 保证任何修改都经过校验 |
| `__check()` 私有方法 | 内部复用校验逻辑 |

本章的继承体系中，**父类负责封装共性**（如 `Vehicle.plate`、`Vehicle.speed`），**子类只重写差异**——这就是封装与继承的配合方式。

### 12.2.2 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 子类直接改父类私有属性 | `AttributeError`（名称已被改写为 `_父类名__x`） | 用父类提供的 `get` / `set` |
| 父类属性全设私有 | 子类无法参与计算，只能靠接口绕 | 共性数据用公有属性，敏感数据用私有 |
| 忘记在 `__init__` 中校验 | 非法初始值进入对象 | 构造末尾调用 `__check()` |

### 12.2.4 随堂示例：封装：把约束放进方法里

**核心要点**：验证封装的价值：外部只调方法、不碰属性，区间约束（0~120）只在类内写一次，任何调用方都无法绕过。

In [ ]:
class Vehicle:
    def __init__(self, plate, speed=0):
        self.plate = plate
        self.__speed = speed

    def accelerate(self, delta):
        self.__speed = min(self.__speed + delta, 120)   # 上限约束在类内
        return self.__speed

    def get_speed(self):
        return self.__speed

v = Vehicle("京A12345", 100)
print("初始车速：", v.get_speed())
print("加速 50：", v.accelerate(50), "（被限制在 120）")
print("再加速 30：", v.accelerate(30))

**预期输出**：

```text
初始车速： 100
加速 50： 120 （被限制在 120）
再加速 30： 120
```

## 12.3 知识点 12.2　继承与 `super()`

### 12.3.1 核心概念

```text
class Vehicle:            # 父类（基类）
    ...

class Truck(Vehicle):     # 子类（派生类）：括号内写父类名
    ...
```

- 子类**自动获得**父类的全部公有属性与方法；
- 子类可新增自己的属性与方法；
- `super()` 用于**调用父类的同名方法**，最常用的是 `super().__init__(...)`。

### 12.3.2 语法要点

| 场景 | 写法 | 说明 |
|---|---|---|
| 单继承 | `class Bus(Vehicle):` | 括号内是父类 |
| 调用父类构造 | `super().__init__(plate, speed)` | **必须在使用 `self` 之前** |
| 调用父类方法 | `super().fee(km, rate)` | 取父类实现的结果再加工 |
| 判断继承关系 | `isinstance(t, Vehicle)` | 子类实例也是父类实例 |

**多继承**：`class A(B, C):` 合法，按 MRO 顺序查找方法，但本章不展开——交通场景中单继承已足够表达"车型 / 设备 / 车道"的分类。

### 12.3.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 子类 `__init__` 忘记 `super()` | `AttributeError: 'Truck' object has no attribute 'plate'` | 首行调用 `super().__init__(...)` |
| `super()` 放在赋值之后 | 可能覆盖已设值，逻辑混乱 | 放首行 |
| 父类名硬编码 `Vehicle.__init__(self, ...)` | 改父类名要改多处 | 用 `super()` |
| 误以为子类会"复制"父类代码 | 实际是"查找" | 理解方法解析顺序 |

### 12.3.4 随堂示例：继承：子类复用父类能力

**核心要点**：验证 `class 子类(父类)` 自动获得父类方法，`super().__init__()` 复用父类初始化；`isinstance()` 认定子类对象同时属于父类。

In [ ]:
class Vehicle:
    def __init__(self, plate):
        self.plate = plate

    def limit(self):
        return 120

    def info(self):
        return self.plate + " 限速 " + str(self.limit()) + " km/h"

class Truck(Vehicle):
    def __init__(self, plate, load):
        super().__init__(plate)         # 复用父类初始化
        self.load = load

    def limit(self):
        return 80                       # 重写：货车限速更低

t = Truck("京B67890", 18)
print(t.info())
print("载重：", t.load, "吨")
print("是 Vehicle 的实例吗：", isinstance(t, Vehicle))

**预期输出**：

```text
京B67890 限速 80 km/h
载重： 18 吨
是 Vehicle 的实例吗： True
```

## 12.4 知识点 12.3　方法重写

### 12.4.1 核心概念

子类定义**与父类同名的方法**即"重写"（override）：调用时优先使用子类版本，父类版本仍可通过 `super()` 调回。

重写有三种典型用法：

| 用法 | 说明 | 案例 |
|---|---|---|
| 完全替换 | 子类给出全新实现 | `Car.limit()` 返回 120 |
| 扩展 | `super().方法()` 取父类结果再加工 | `Truck.fee()` 乘 1.8 |
| 追加信息 | 父类字符串 + 子类补充 | `Truck.info()` 加"载重" |

### 12.4.2 语法要点

- 方法名、参数表应与父类**保持一致**（参数可增加默认值，但不建议改顺序）；
- `super().info()` 只能取到父类实现，不会再次触发子类的 `info()`（避免无限递归）；
- 重写后父类方法依然存在，可用 `父类名.方法(obj)` 显式调用（不推荐）。

### 12.4.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 子类方法名拼错 | 变成新方法，父类版本仍被调用 | 对照父类签名检查 |
| `super().fee(km)` 漏参数 | `TypeError` | 参数与父类一致 |
| 在 `__str__` 中调 `self.info()` | 逻辑绕、易递归 | 直接用 `super().info()` |
| 重写后忘记返回 | 调用方得到 `None` | 每个分支都 `return` |

### 12.4.4 随堂示例：方法重写：替换与扩展

**核心要点**：验证重写的两种形态：完全替换（直接给新实现）与扩展（先 `super()` 取父类结果再加工），二者对外接口保持一致。

In [ ]:
class Vehicle:
    def fee(self, km, rate=0.5):
        return round(km * rate, 2)

class Bus(Vehicle):
    def fee(self, km, rate=0.5):
        return round(km * rate * 1.2, 2)     # 替换：直接新实现

class Truck(Vehicle):
    def fee(self, km, rate=0.5):
        base = super().fee(km, rate)         # 扩展：先取父类结果
        return round(base * 1.8, 2)

print("小客车 120km：", Vehicle().fee(120))
print("客车   120km：", Bus().fee(120))
print("货车   120km：", Truck().fee(120))

**预期输出**：

```text
小客车 120km： 60.0
客车   120km： 72.0
货车   120km： 108.0
```

## 12.5 知识点 12.4　多态与接口约定

### 12.5.1 核心概念

**多态**：同一句调用，作用在不同类型的对象上产生不同行为。前提是"这些对象共用同一个方法名"——这个共同的方法名就是**接口**。

```text
for d in devices:
    print(d.status())      # 不判断类型，各自实现
```

`poll()` 完全不知道设备是灯、情报板还是道闸，新增"毫米波雷达"时**只需新增一个子类**，`poll()` 一行都不用改。

**接口约定**：基类方法直接 `raise NotImplementedError("子类必须实现…")`，强制子类重写。若子类忘记实现，调用时立刻报错，而不是静默返回错误结果。

### 12.5.2 语法要点

| 写法 | 作用 |
|---|---|
| 基类 `def status(self): raise NotImplementedError(...)` | 声明"必须由子类实现" |
| 子类 `def status(self): return "…"` | 提供具体实现 |
| 统一循环 `for d in devices: d.status()` | 调用方不判断类型 |

### 12.5.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| `poll()` 里写 `isinstance` 分支 | 退化为第 3 章写法，失去多态 | 只依赖共同方法名 |
| 子类漏实现基类方法 | 调用时 `NotImplementedError` | 补齐实现 |
| 方法名不一致（`get_status` / `status`） | 多态失效 | 统一命名 |
| 基类方法返回 `None` 而非抛错 | 静默错误，难发现 | 用 `raise NotImplementedError` |

### 12.5.4 随堂示例：多态：同一接口，不同实现

**核心要点**：验证多态：调用方只写 `device.run()`，具体执行哪段代码由对象类型决定，因此新增设备时主循环无需改动。

In [ ]:
class Device:
    def run(self):
        raise NotImplementedError("子类必须实现 run()")

class Light(Device):
    def run(self):
        return "信号灯：切换相位"

class Board(Device):
    def run(self):
        return "情报板：刷新提示"

class Barrier(Device):
    def run(self):
        return "道闸：抬杆放行"

for d in [Light(), Board(), Barrier()]:
    print(d.run())          # 同一句调用，各自执行

try:
    Device().run()          # 基类未实现，直接调用会报错
except NotImplementedError as e:
    print("基类直接调用：", e)

**预期输出**：

```text
信号灯：切换相位
情报板：刷新提示
道闸：抬杆放行
基类直接调用： 子类必须实现 run()
```

## 12.6 知识点 12.5　`isinstance()` 与 `type()`

### 12.6.1 核心概念

| 函数 | 判断内容 | 子类实例的结果 |
|---|---|---|
| `isinstance(obj, 类)` | 是不是该类**或其子类**的实例 | `True` |
| `type(obj) is 类` | 类型**精确**等于该类 | `False` |

因此在"按类型分支"时，`isinstance` 更符合直觉（货车"是"车辆）；`type()` 用于需要精确判定的场景（如日志记录精确类型名）。

`isinstance` 的第二个参数可以是**元组**：`isinstance(x, (int, float))` 表示"任一类型即可"。

### 12.6.2 语法要点

| 写法 | 结果（`t = Truck(...)`） |
|---|---|
| `isinstance(t, Truck)` | `True` |
| `isinstance(t, Vehicle)` | `True` |
| `type(t) is Vehicle` | `False` |
| `type(t).__name__` | `"Truck"` |

### 12.6.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 用 `type(x) == Vehicle` 判子类 | 子类被判为"不是车辆" | 用 `isinstance` |
| `isinstance` 第二参数写成列表 | `TypeError` | 用元组 |
| 过度使用类型判断 | 代码僵硬，新增类型要改多处 | 优先用多态 |

---

### 12.6.4 随堂示例：isinstance 与 type 的取舍

**核心要点**：验证二者差异：`isinstance()` 沿继承链判断（子类对象也算父类），`type()` 只认精确类型；接口判断用前者更安全。

In [ ]:
class Vehicle:
    pass

class Truck(Vehicle):
    pass

t = Truck()
print("type(t).__name__：", type(t).__name__)
print("isinstance(t, Truck)：", isinstance(t, Truck))
print("isinstance(t, Vehicle)：", isinstance(t, Vehicle))
print("type(t) is Vehicle：", type(t) is Vehicle)

items = [Vehicle(), Truck()]
for x in items:
    print(type(x).__name__, "是 Vehicle 吗：", isinstance(x, Vehicle))

**预期输出**：

```text
type(t).__name__： Truck
isinstance(t, Truck)： True
isinstance(t, Vehicle)： True
type(t) is Vehicle： False
Vehicle 是 Vehicle 吗： True
Truck 是 Vehicle 吗： True
```

# 第二部分　代码示范

## 案例 12-1　车型继承体系与方法重写

- **覆盖知识点**：12.2（继承、`super()`）、12.3（重写：替换 / 扩展 / 追加）、12.5（`isinstance`）
- **源文件**：`case1_车型继承.py`
- **难度**：基础　**建议用时**：30 分钟

### 任务情境

收费与限速管理需要区分小客车、货车、客车三类。三者共享车牌、车速、基础费率，但在限速、费率系数与信息描述上有差异。

### 任务要求

1. 定义父类 `Vehicle`，含 `plate`、`speed`、`limit()`、`fee(km)`、`info()`；
2. 定义子类 `Car`、`Truck`、`Bus`，各自重写 `limit()`；
3. `Truck` 需扩展载重属性，用 `super().__init__()` 复用父类初始化；
4. `Truck.fee()` 用 `super().fee(km)` 取基础费用后上浮；
5. 用同一个循环调用三种车型的 `info()` 与 `fee()`。

### 参考代码

In [ ]:
# ---------- 第 12 章 面向对象编程（下）· 案例 12-1：车型继承体系与方法重写 ----------
# 对应知识点：12.2 继承与 super()、12.3 方法重写（替换 / 扩展）、12.5 isinstance() 与 type()
# 功能说明：由车辆基类派生出小客车、货车与客车，重写限速、计费与信息输出方法

class Vehicle:
    """车辆基类"""

    def __init__(self, plate, speed=0):
        self.plate = plate
        self.speed = speed

    def limit(self):
        """限速，默认 120"""
        return 120

    def fee(self, km, rate=0.5):
        """基础通行费"""
        return round(km * rate, 2)

    def info(self):
        return self.plate + " 限速 " + str(self.limit()) + " km/h"


class Car(Vehicle):
    """小客车"""

    def limit(self):
        return 120


class Truck(Vehicle):
    """货车：新增载重，限速更低，费率上浮"""

    def __init__(self, plate, load, speed=0):
        super().__init__(plate, speed)     # 复用父类初始化
        self.load = load

    def limit(self):
        return 80

    def fee(self, km, rate=0.5):
        base = super().fee(km, rate)       # 先算基础费用
        return round(base * 1.8, 2)        # 货车按 1.8 倍计费

    def info(self):
        return super().info() + "，载重 " + str(self.load) + " 吨"


class Bus(Vehicle):
    """客车"""

    def limit(self):
        return 100

    def fee(self, km, rate=0.5):
        return round(km * rate * 1.2, 2)


fleet = [Car("京A12345"), Truck("京B67890", 18), Bus("京C24680"), Car("京D13579")]

for v in fleet:
    print(v.info(), "| 行驶 120km 通行费", v.fee(120), "元")

print()
print("是否货车：", isinstance(fleet[1], Truck))
print("是否车辆：", isinstance(fleet[1], Vehicle))
print("类型：", type(fleet[1]).__name__)

### 运行结果

```text
京A12345 限速 120 km/h | 行驶 120km 通行费 60.0 元
京B67890 限速 80 km/h，载重 18 吨 | 行驶 120km 通行费 108.0 元
京C24680 限速 100 km/h | 行驶 120km 通行费 72.0 元
京D13579 限速 120 km/h | 行驶 120km 通行费 60.0 元

是否货车： True
是否车辆： True
类型： Truck
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `class Car(Vehicle):` | 12.2 | 圆括号写父类，自动获得 `__init__`、`fee`、`info` |
| `Car.limit()` 返回 120 | 12.3 | 完全替换：与父类相同也显式写出，表明"这是本车型的规则" |
| `Truck.__init__` 首行 `super().__init__(...)` | 12.2 | 复用父类对 `plate` / `speed` 的初始化 |
| `self.load = load` | 12.2 | 子类扩展属性，父类不知道它存在 |
| `super().fee(km, rate)` | 12.3 | 扩展：先取父类结果，再乘 1.8 |
| `super().info() + "，载重 …"` | 12.3 | 追加信息：父类描述 + 子类补充 |
| `Bus.fee()` 完全重写 | 12.3 | 不调用父类，直接按 1.2 倍计算 |
| `for v in fleet:` 统一调用 | 12.4 | 多态的雏形：不判断类型，各自执行 |
| `isinstance(…, Vehicle)` | 12.5 | 子类实例也属于父类 |

**改一改**：删掉 `Truck.__init__` 中的 `super().__init__(plate, speed)`，运行后记录 `AttributeError` 出现在哪一行；再把 `Truck.info()` 改成 `self.plate + " 载重 " + str(self.load)`，观察 `info()` 与 `super().info()` 的信息差异。

## 案例 12-2　多态——统一调度不同路侧设备

- **覆盖知识点**：12.4（多态、`NotImplementedError` 接口约定）、12.2（`super().__init__`）、12.3（重写）
- **源文件**：`case2_设备多态.py`
- **难度**：综合　**建议用时**：35 分钟

### 任务情境

路侧设备包括信号灯、可变情报板、道闸三类。巡检程序需要"一次性遍历所有设备并读取状态、下发刷新指令"，但每类设备的指令细节不同。

### 任务要求

1. 定义基类 `SignalDevice`，`status()` 与 `refresh()` 直接 `raise NotImplementedError`，约定子类必须实现；
2. 三个子类分别实现自己的状态描述与刷新动作；
3. 编写 `poll(devices)` 函数，用同一个 `for` 循环调用 `status()` 与 `refresh()`，**不判断类型**；
4. 输出巡检汇总。

### 参考代码

In [ ]:
# ---------- 第 12 章 面向对象编程（下）· 案例 12-2：多态——统一调度不同路侧设备 ----------
# 对应知识点：12.4 多态与 NotImplementedError 接口约定、12.2 super().__init__、12.3 重写
# 功能说明：用基类约定统一接口，循环调度信号灯、情报板、道闸等设备，各自实现自身行为

class SignalDevice:
    """路侧设备基类：约定接口"""

    def __init__(self, code):
        self.code = code

    def status(self):
        raise NotImplementedError("子类必须实现 status()")

    def refresh(self):
        raise NotImplementedError("子类必须实现 refresh()")


class Light(SignalDevice):
    """信号灯"""

    def __init__(self, code, phase="红灯"):
        super().__init__(code)
        self.phase = phase

    def status(self):
        return "信号灯 " + self.code + " 当前相位 " + self.phase

    def refresh(self):
        return "信号灯 " + self.code + " 已同步配时方案"


class VariableSign(SignalDevice):
    """可变情报板"""

    def __init__(self, code, text="限速 60"):
        super().__init__(code)
        self.text = text

    def status(self):
        return "情报板 " + self.code + " 显示 <" + self.text + ">"

    def refresh(self):
        return "情报板 " + self.code + " 已刷新显示内容"


class Barrier(SignalDevice):
    """道闸"""

    def __init__(self, code, opened=False):
        super().__init__(code)
        self.opened = opened

    def status(self):
        state = "抬起" if self.opened else "落下"
        return "道闸 " + self.code + " 当前" + state

    def refresh(self):
        self.opened = not self.opened
        return "道闸 " + self.code + " 已切换为" + ("抬起" if self.opened else "落下")


def poll(devices):
    """统一巡检：不判断具体类型"""
    print("=== 设备巡检开始，共 " + str(len(devices)) + " 台 ===")
    for d in devices:
        print("[状态]", d.status())
        print("[指令]", d.refresh())
    print("=== 巡检结束 ===")


devices = [
    Light("L-01", "绿灯"),
    VariableSign("V-07", "前方事故，减速慢行"),
    Barrier("B-03"),
    Light("L-02", "黄灯"),
]

poll(devices)
print()
print("第二轮巡检（道闸状态已翻转）：")
poll(devices)

### 运行结果

```text
=== 设备巡检开始，共 4 台 ===
[状态] 信号灯 L-01 当前相位 绿灯
[指令] 信号灯 L-01 已同步配时方案
[状态] 情报板 V-07 显示 <前方事故，减速慢行>
[指令] 情报板 V-07 已刷新显示内容
[状态] 道闸 B-03 当前落下
[指令] 道闸 B-03 已切换为抬起
[状态] 信号灯 L-02 当前相位 黄灯
[指令] 信号灯 L-02 已同步配时方案
=== 巡检结束 ===

第二轮巡检（道闸状态已翻转）：
=== 设备巡检开始，共 4 台 ===
[状态] 道闸 B-03 当前抬起
...
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| 基类两个 `raise NotImplementedError` | 12.4 | 接口约定：子类必须实现，漏实现立刻报错 |
| `super().__init__(code)` | 12.2 | 三个子类共用编号初始化，避免重复代码 |
| 子类各自实现 `status` / `refresh` | 12.4 | 同类方法名、不同行为，这就是多态的前提 |
| `poll()` 中 `for d in devices` | 12.4 | 不判断类型，只依赖共同方法名 |
| `"抬起" if self.opened else "落下"` | 12.4、3.2 | 条件表达式，省去 `if-else` 块 |
| `self.opened = not self.opened` | 12.4 | `refresh` 带副作用：翻转状态，二次巡检结果因此不同 |

**改一改**：把 `Barrier.status` 改名为 `get_status`，运行后记录 `NotImplementedError`（父类版本被调用）；再新建一个 `class Radar(SignalDevice):` 只实现 `status`，调用 `poll()` 时观察在哪一步报错——验证接口约定的"强制"效果。

---

# 第三部分　布置练习

## 练习 12-1　ETC 车道与人工车道

- **对应知识点**：12.2（继承、`super().__init__`）、12.3（重写：通行能力、计费、信息追加）
- **提交文件**：`practice1_车道继承.py`
- **难度**：综合　**建议用时**：30 分钟

### 题目

收费站有两种车道：ETC 车道与人工车道。二者都有车道编号，但通行能力不同（ETC 1500 辆/h、人工 400 辆/h），收费折扣也不同（ETC 95 折、人工无折扣）。

### 要求

1. 定义父类 `Lane`，含编号、`capacity()`（默认 400）、`fee(km)`（默认无折扣）；
2. 定义 `EtcLane` 与 `ManualLane`，分别重写通行能力与计费；
3. `EtcLane` 增加 `device_ok` 属性，设备故障时拒绝收费并提示；
4. 用统一循环输出两类车道的通行能力与 200km 费用，比较差异。

### 提示

- `super().fee(km)` 可先取到无折扣费用，再乘 0.95。
- 子类 `__init__` 中如需新增属性，务必先调用 `super().__init__(...)`。
- 设备故障时"拒绝收费"应返回一个可判断的值（如 `-1`），并在 `info()` 中显式标注状态。

### 自检要点（不含答案）

- [ ] ETC 200km 费用为 95.0 元，人工为 100.0 元；
- [ ] ETC 与人工的通行能力比为 3.75；
- [ ] 故障 ETC 车道的 `info()` 显示"设备故障"；
- [ ] `isinstance(lanes[0], Lane)` 为 `True`。

## 练习 12-2　充电设施的多态计费

- **对应知识点**：12.4（多态、基类统一实现）、12.3（重写）、第 6 章（模板方法思想）
- **提交文件**：`practice2_充电桩多态.py`
- **难度**：进阶　**建议用时**：35 分钟

### 题目

智慧停车场内有三类充电桩：慢充（7kW）、快充（60kW）、超充（250kW）。计费规则为"电量 × 单价 + 服务费"，单价与超时占位费各不相同。要求用多态实现统一结算。

### 要求

1. 定义基类 `Charger`，含编号、`power()`、`price()`、`settle(kwh, minutes)`；
2. 三个子类重写 `power()` 与 `price()`，`settle()` 在基类中统一实现（模板方法思想）；
3. 加一个 `overtime_fee(minutes)`，基类返回 0，超充桩重写为超过 60 分钟按 2 元/分钟收取；
4. 遍历三类充电桩结算同一笔"35 kWh、85 分钟"的订单。

### 提示

- 父类 `settle()` 里调用 `self.price()` 与 `self.overtime_fee()`，运行时会自动绑定到**子类**的实现——这就是多态在方法内部同样生效。
- 服务费按电费的 10% 计算，四舍五入到分（2 位小数）。
- 超时费只对超充桩生效，慢充与快充沿用基类返回 0，不必重写。

### 自检要点（不含答案）

- [ ] 三个子类只需重写 `power` 与 `price`，`settle` 一律继承；
- [ ] 35 kWh/85 min 的订单：慢充 23.1 元、快充 38.5 元、超充 111.6 元；
- [ ] 超充超时费 50 元，慢充 0 元；
- [ ] `settle()` 内不出现 `if 类型` 判断。

---

# 第四部分　习题答案

## 答案 12-1　ETC 车道与人工车道

> 对应 **练习 12-1**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 12 章 面向对象编程（下）· 练习 12-1 参考答案：ETC 车道与人工车道 ----------
# 对应知识点：12.2 继承与 super().__init__、12.3 重写（通行能力 / 计费 / 信息追加）
# 功能说明：由车道基类派生 ETC 车道与人工车道，分别重写通行能力与收费规则

class Lane:
    """车道基类"""

    def __init__(self, code):
        self.code = code

    def capacity(self):
        return 400

    def fee(self, km, rate=0.5):
        return round(km * rate, 2)

    def info(self):
        return self.code + " 通行能力 " + str(self.capacity()) + " 辆/h"


class EtcLane(Lane):
    """ETC 车道"""

    def __init__(self, code, device_ok=True):
        super().__init__(code)
        self.device_ok = device_ok

    def capacity(self):
        return 1500

    def fee(self, km, rate=0.5):
        return round(super().fee(km, rate) * 0.95, 2)

    def info(self):
        state = "正常" if self.device_ok else "故障"
        return super().info() + "，设备" + state


class ManualLane(Lane):
    """人工车道"""

    def capacity(self):
        return 400

    def info(self):
        return super().info() + "，人工发卡"


lanes = [EtcLane("E-01"), EtcLane("E-02", device_ok=False), ManualLane("M-01")]

for lane in lanes:
    print(lane.info(), "| 200km 费用", lane.fee(200), "元")

print()
print("ETC 与人工通行能力比：", round(EtcLane("X").capacity() / ManualLane("Y").capacity(), 2))
print("是否车道子类：", isinstance(lanes[0], Lane))

### 运行结果

```text
E-01 通行能力 1500 辆/h，设备正常 | 200km 费用 95.0 元
E-02 通行能力 1500 辆/h，设备故障 | 200km 费用 95.0 元
M-01 通行能力 400 辆/h，人工发卡 | 200km 费用 100.0 元

ETC 与人工通行能力比： 3.75
是否车道子类： True
```

### 解析

1. **重写的三种用法齐备**：`capacity` 是完全替换（1500 vs 400）；`fee` 是扩展（父类结果 × 0.95）；`info` 是追加（父类描述 + 设备状态）。
2. **`super().fee(km, rate)` 的位置**：必须先拿到父类结果再乘折扣。若写成 `round(km * rate * 0.95, 2)`，则与父类的计费公式重复实现，父类改费率时子类会失同步。
3. **故障车道仍能算费**：本题只要求"标注状态"，因此 `fee()` 未对 `device_ok` 做拦截；真实系统中应在 `fee()` 内判断 `device_ok` 并返回 `-1` 或抛异常——这是"业务规则放哪一层"的设计问题，值得课堂讨论。
4. **`isinstance(lanes[0], Lane)`**：`EtcLane` 实例也是 `Lane` 实例，这正是多态能成立的原因。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| `AttributeError: code` | `EtcLane.__init__` 漏 `super()` | 补 `super().__init__(code)` |
| 费用 190.0 | 折扣乘了两次 | 只乘一次 0.95 |
| 人工车道也打 95 折 | `ManualLane` 继承了 `EtcLane` | `ManualLane(Lane)` |
| 比值 3.75 变成 0.27 | 分子分母颠倒 | ETC ÷ 人工 |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 继承结构 | 3 | 两个子类均继承 `Lane` |
| 通行能力与计费 | 4 | 1500/400、95.0/100.0 |
| `super()` 使用 | 2 | 构造与计费各一次 |
| 状态标注 | 1 | 故障车道显示"设备故障" |

## 答案 12-2　充电设施的多态计费

> 对应 **练习 12-2**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 12 章 面向对象编程（下）· 练习 12-2 参考答案：充电设施的多态计费 ----------
# 对应知识点：12.4 多态与基类统一实现、12.3 方法重写、6.x 模板方法思想
# 功能说明：由充电桩基类派生快充与慢充，用统一接口按不同电价与功率计算充电费用

class Charger:
    """充电桩基类"""

    def __init__(self, code):
        self.code = code

    def power(self):
        return 7

    def price(self):
        return 0.6

    def overtime_fee(self, minutes):
        return 0

    def settle(self, kwh, minutes):
        energy_fee = round(kwh * self.price(), 2)
        service = round(energy_fee * 0.1, 2)
        ot = self.overtime_fee(minutes)
        return round(energy_fee + service + ot, 2)

    def info(self):
        return self.code + " " + str(self.power()) + "kW 单价 " + str(self.price()) + " 元/kWh"


class SlowCharger(Charger):
    def power(self):
        return 7

    def price(self):
        return 0.6


class FastCharger(Charger):
    def power(self):
        return 60

    def price(self):
        return 1.0


class SuperCharger(Charger):
    def power(self):
        return 250

    def price(self):
        return 1.6

    def overtime_fee(self, minutes):
        if minutes > 60:
            return (minutes - 60) * 2
        return 0


chargers = [SlowCharger("C-01"), FastCharger("C-02"), SuperCharger("C-03")]

for c in chargers:
    total = c.settle(35, 85)
    print(c.info(), "| 35kWh/85min 应收", total, "元")

print()
print("超充超时费：", SuperCharger("C-03").overtime_fee(85), "元")
print("慢充超时费：", SlowCharger("C-01").overtime_fee(85), "元")

### 运行结果

```text
C-01 7kW 单价 0.6 元/kWh | 35kWh/85min 应收 23.1 元
C-02 60kW 单价 1.0 元/kWh | 35kWh/85min 应收 38.5 元
C-03 250kW 单价 1.6 元/kWh | 35kWh/85min 应收 111.6 元

超充超时费： 50 元
慢充超时费： 0 元
```

### 解析

1. **模板方法**：`settle()` 只在基类写一次，内部依次调用 `self.price()` 与 `self.overtime_fee()`。运行时 `self` 是子类实例，因此自动绑定到子类实现——**多态不仅在调用方生效，在类内部同样生效**。
2. **计算核对（超充）**：电费 35 × 1.6 = 56.0；服务费 56 × 10% = 5.6；超时费 (85 − 60) × 2 = 50；合计 111.6 元。
3. **未重写即继承**：慢充、快充不写 `overtime_fee`，直接继承"返回 0"，因此无超时费。这体现"子类只写差异"。
4. **新增桩型的成本**：若增加"换电站"，只需新建子类并重写 `power` / `price`，`settle()` 与主循环都无需改动——这就是多态的可扩展性。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 超充 61.6 元 | 漏了超时费 | `settle` 加 `self.overtime_fee()` |
| 慢充也收超时费 | `overtime_fee` 在基类里写了 2 元/分钟 | 基类返回 0 |
| 三个桩结果相同 | `settle` 里写死单价 | 用 `self.price()` |
| 服务费按 10 元固定 | 混淆比例与定额 | `energy_fee * 0.1` |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 基类结构 | 3 | 四个方法，`settle` 为模板 |
| 子类重写 | 4 | 功率、单价、超时费各自正确 |
| 结算结果 | 2 | 23.1 / 38.5 / 111.6 |
| 多态调用 | 1 | 主循环无类型判断 |

---

## 本章小结

- **封装**：私有属性 + 公有 `get` / `set`，校验写在唯一入口；子类通过接口访问父类私有数据。
- **继承**：`class 子类(父类)`；子类 `__init__` 首行 `super().__init__(...)`。
- **重写**：同名方法覆盖父类，三种用法——完全替换、`super()` 扩展、追加信息。
- **多态**：同一方法名、不同实现；调用方只依赖接口，新增类型不改调用方。
- **接口约定**：基类 `raise NotImplementedError` 强制子类实现，漏实现立即报错。
- **类型判断**：`isinstance` 含子类，`type()` 精确匹配；能不用类型判断就不用。

## 自检清单

- [ ] 能写出"父类 + 两个子类"的最小继承结构，并在子类中正确调用 `super().__init__()`；
- [ ] 能说明"重写"与"调用父类同名方法"如何共存；
- [ ] 能用 `NotImplementedError` 约定接口，并解释其价值；
- [ ] 能举一个多态的例子，说明"新增类型不用改调用方"；
- [ ] 知道 `isinstance` 与 `type()` 的区别；
- [ ] 能识别"用 `isinstance` 分支替代多态"的反模式。

## 前后章节衔接

- 本章的继承体系是**第 13 章综合项目**的组织基础（车辆、设备、场站都可建模为类）。
- 本章的"接口约定 + 统一遍历"正是 `pygame` 精灵组更新、`turtle` 多海龟绘制所依赖的思路。
- 本章的重写在**终结性考核任务三**中直接考到：`EmergencyPlan` 重写 `green_time()` 并用 `super()` 取父类值。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：面向对象编程上](../11面向对象编程上/11_面向对象编程上.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[下一章：综合项目实战 ▶](../13综合项目实战/13_综合项目实战.ipynb)